In [ ]:
!pip install bitsandbytes
!pip install accelerate
!pip install transformers
!pip install peft
!pip install datasets
!pip install bitsandbytes accelerate transformers peft datasets
!pip install boto3
# === Chunk 0: Credentials & Common Imports ===

# Replace the 'XXX' strings with your real tokens/keys before pushing to GitHub
import os

os.environ["HF_TOKEN"] = "XXX_HUGGINGFACE_TOKEN_XXX"
os.environ["AWS_ACCESS_KEY_ID"] = "XXX_AWS_ACCESS_KEY_ID_XXX"
os.environ["AWS_SECRET_ACCESS_KEY"] = "XXX_AWS_SECRET_ACCESS_KEY_XXX"

# Common imports
import json
import boto3
import warnings
warnings.filterwarnings("ignore", message=".*use_auth_token.*")

# HF login (optional if HF_TOKEN is set)
from huggingface_hub import login
login(token=os.getenv("HF_TOKEN"))


In [ ]:
# === Chunk 1: Download from S3 & Prepare Dataset ===

# 1. Download train.jsonl from S3
bucket = "my-qa-dataset"
key = "data/train.jsonl"
local_train = "./train.jsonl"

s3 = boto3.client(
    "s3",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
)
s3.download_file(bucket, key, local_train)

# 2. Read first 2000 lines into a HuggingFace Dataset
from datasets import Dataset

data_list = []
with open(local_train, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 2000:
            break
        obj = json.loads(line)
        data_list.append({"text": obj["text"]})

raw_ds = Dataset.from_list(data_list)

# 3. Tokenize and remove 'text' column
from transformers import AutoTokenizer

model_name = "meta-llama/Meta-Llama-3-8B"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True,
    use_auth_token=os.getenv("HF_TOKEN")
)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(ex):
    return tokenizer(
        ex["text"],
        max_length=512,
        padding="max_length",
        truncation=True
    )

tokenized = raw_ds.map(tokenize_fn, batched=True)
tokenized = tokenized.remove_columns(["text"])


In [ ]:
# === Chunk 2: Fine‑tune with QLoRA & Upload Adapter to S3 ===

import torch, os, json, boto3, warnings
warnings.filterwarnings("ignore", message=".*use_auth_token.*")

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling, BitsAndBytesConfig
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# 1. Credentials & HF login
# (already set in chunk 0)

# 2. Load and quantize base model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    use_auth_token=os.getenv("HF_TOKEN")
)
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# 3. Apply LoRA
lora_cfg = LoraConfig(
    r=64,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_cfg)

# 4. Trainer setup
training_args = TrainingArguments(
    output_dir="./qlora-output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    bf16=True,
    logging_strategy="steps",
    logging_steps=10,
    save_steps=2000,
    save_total_limit=1,
    report_to="none",
    remove_unused_columns=False
)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator
)

# 5. Run training
trainer.train()

# 6. Save LoRA adapter locally
adapter_dir = "./qlora-output/adapter"
trainer.model.save_pretrained(adapter_dir)

# 7. Upload that adapter to S3 (exactly as your original code)
s3 = boto3.client(
    "s3",
    aws_access_key_id    = os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key= os.getenv("AWS_SECRET_ACCESS_KEY"),
)
bucket = "my-qa-dataset"
prefix = "lora-adapters/"  # S3 folder

for filename in os.listdir(adapter_dir):
    local_path = os.path.join(adapter_dir, filename)
    s3_key     = prefix + filename
    print(f"Uploading {local_path} → s3://{bucket}/{s3_key}")
    s3.upload_file(local_path, bucket, s3_key)

print("✅ LoRA adapter has been uploaded to S3.")


In [ ]:
# === Chunk 3: Download Adapter from S3, Merge into Base Model, and Save ===

import os, boto3
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Download the adapter files from S3
bucket = "my-qa-dataset"
prefix = "lora-adapters/"      # same folder you uploaded into
local_adapter = "./adapter"    # where we'll store them locally
os.makedirs(local_adapter, exist_ok=True)

s3 = boto3.client(
    "s3",
    aws_access_key_id    = os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key= os.getenv("AWS_SECRET_ACCESS_KEY"),
)
resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
for obj in resp.get("Contents", []):
    key = obj["Key"]
    filename = key.split("/")[-1]
    s3.download_file(bucket, key, os.path.join(local_adapter, filename))
    print(f"Downloaded {filename}")

# 2. Load the base Llama 3 model
model_name = "meta-llama/Meta-Llama-3-8B"
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto",
    use_auth_token=os.getenv("HF_TOKEN")
)

# 3. Load the LoRA adapter into the PEFT wrapper
peft_model = PeftModel.from_pretrained(base_model, local_adapter, is_trainable=False)

# 4. Merge the adapter weights into the base model
merged_model = peft_model.merge_and_unload()

# 5. Save the merged model and tokenizer
merged_dir = "./merged_llama3"
os.makedirs(merged_dir, exist_ok=True)
merged_model.save_pretrained(merged_dir)

# 6. Also save the tokenizer for inference
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_auth_token=os.getenv("HF_TOKEN")
)
tokenizer.save_pretrained(merged_dir)

print("✅ Adapter merged and saved to", merged_dir)


In [ ]:
# === Chunk 4: Inference + BLEU/ROUGE/BERTScore Evaluation ===

!pip install evaluate rouge_score bert_score

# 1. Download eval.jsonl
s3.download_file(bucket, "data/eval.jsonl", "eval.jsonl")

# 2. Read first 10 samples
samples = []
with open("eval.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 10: break
        samples.append(json.loads(line)["text"])

# 3. Load merged model & do inference
from transformers import TextGenerationPipeline, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    merged_dir,
    torch_dtype="auto",
    device_map="auto"
)
pipe = TextGenerationPipeline(model=model, tokenizer=tokenizer)

preds = []
for text in samples:
    instruction = text.split("\nResponse:")[0] + "\nResponse:"
    out = pipe(instruction, max_new_tokens=128, do_sample=False)[0]["generated_text"]
    pred = out[len(instruction):].strip()
    preds.append(pred)
    print("Instruction:", instruction)
    print("Prediction:", pred, "\n")

# 4. BLEU
from evaluate import load
bleu = load("bleu")
refs = [[t.split("\nResponse:")[1].strip()] for t in samples]
print("BLEU:", bleu.compute(predictions=preds, references=refs)["bleu"] * 100)

# 5. ROUGE & BERTScore
rouge = load("rouge")
rs = rouge.compute(predictions=preds, references=[t.split("\nResponse:")[1].strip() for t in samples])
print(f"ROUGE-1: {rs['rouge1']*100:.2f}, ROUGE-2: {rs['rouge2']*100:.2f}, ROUGE-L: {rs['rougeL']*100:.2f}")

bertscore = load("bertscore")
bs = bertscore.compute(
    predictions=preds,
    references=[t.split("\nResponse:")[1].strip() for t in samples],
    lang="en"
)
print("BERTScore F1:", sum(bs["f1"]) / len(bs["f1"]) * 100)
